In [1]:
%load_ext autoreload
%autoreload 2
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from scentree.io.writer import save_json
from scentree.io.loader import Dataset, DatasetsLoader
from scentree.fan_generator import StageManager
from scentree.tree_construction.ftc import FTC

logging.basicConfig(level=logging.INFO)

SEED = 42
np.random.seed(SEED)  # global seed: FTC.generate_scenario_trees() uses np.random internally, no seed param

ImportError: cannot import name 'Self' from 'typing' (/usr/lib/python3.10/typing.py)

# Real data

### Loader

In [43]:
is_15 = True
if is_15:
    data_folder = Path("data_15min")
else:
    data_folder = Path("data_60min")
dam = pd.read_csv(data_folder / "DA.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
rm = pd.read_csv(data_folder / "RM.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
im1 = pd.read_csv(data_folder / "IM1.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
im2 = pd.read_csv(data_folder / "IM2.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
wind = pd.read_csv(data_folder / "WP.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
solar = pd.read_csv(data_folder / "PV.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
im3 = pd.read_csv(data_folder / "IM3.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
ib_up = pd.read_csv(data_folder / "IB_UP.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
ib_down = pd.read_csv(data_folder / "IB_DOWN.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
print(dam.shape, rm.shape, im1.shape, im2.shape, wind.shape, solar.shape, im3.shape, ib_up.shape, ib_down.shape)

(485, 96) (485, 96) (485, 96) (485, 96) (485, 96) (485, 96) (485, 48) (485, 96) (485, 96)


In [ ]:
dam = dam[dam.index < "2025-12-01"]
rm = rm[rm.index < "2025-12-01"]
im1 = im1[im1.index < "2025-12-01"]
wind = wind[wind.index < "2025-12-01"]
im2 = im2[im2.index < "2025-12-01"]
solar = solar[solar.index < "2025-12-01"]
im3 = im3[im3.index < "2025-12-01"]
ib_up = ib_up[ib_up.index < "2025-12-01"]
ib_down = ib_down[ib_down.index < "2025-12-01"]

In [45]:
if is_15:
    renewable_stages = [i for i in range(5, 30) if i != 15 for _ in range(4)]
else:
    renewable_stages = [i for i in range(5, 30) if i != 15 for _ in range(1)]
print(renewable_stages)

[5, 5, 5, 5, 6, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 8, 9, 9, 9, 9, 10, 10, 10, 10, 11, 11, 11, 11, 12, 12, 12, 12, 13, 13, 13, 13, 14, 14, 14, 14, 16, 16, 16, 16, 17, 17, 17, 17, 18, 18, 18, 18, 19, 19, 19, 19, 20, 20, 20, 20, 21, 21, 21, 21, 22, 22, 22, 22, 23, 23, 23, 23, 24, 24, 24, 24, 25, 25, 25, 25, 26, 26, 26, 26, 27, 27, 27, 27, 28, 28, 28, 28, 29, 29, 29, 29]


In [46]:
datasets = [
    Dataset(
        name="DA",
        values=dam.values,
        stage_ids=[1] * dam.shape[1],
    ),
    Dataset(
        name="RM",
        values=rm.values,
        stage_ids=[2] * rm.shape[1],
    ),
    Dataset(
        name="IM1",
        values=im1.values,
        stage_ids=[3] * im1.shape[1],
    ),
    Dataset(
        name="IM2",
        values=im2.values,
        stage_ids=[4] * im2.shape[1],
    ),
    Dataset(
        name="WP",
        values=wind.values,
        stage_ids=renewable_stages,
        bounds=(0,1),
    ),
    Dataset(
        name="PV",
        values=solar.values,
        stage_ids=renewable_stages,
        bounds=(0,1),
    ),
    Dataset(
        name="IM3",
        values=im3.values,
        stage_ids=[15] * im3.shape[1],
    ),
    Dataset(
        name="IB_UP",
        values=ib_up.values,
        stage_ids=[30] * ib_up.shape[1],
    ),
    Dataset(
        name="IB_DOWN",
        values=ib_down.values,
        stage_ids=[30] * ib_down.shape[1],
    ),
]
dataset_loader = DatasetsLoader(datasets=datasets)
full_bounds = dataset_loader.get_full_bounds()
full_values = dataset_loader.get_full_values()
num_variables_per_stage = dataset_loader.get_num_variables_per_stage()
stage_ids = dataset_loader.get_sorted_stage_ids()
map_columns_names = dataset_loader.create_stages_columns_mapping()

### Scenario fan

In [ ]:
# Scenario fan
num_fans = 31
build_in_sample_fans = True
stage_manager = StageManager()
for num_scenarios in range(50,601,10):
    scenario_fans = stage_manager.generate_scenario_fans(
        X=full_values,
        num_fans=num_fans,
        num_scenarios=num_scenarios,
        build_in_sample_fans=build_in_sample_fans,
        value_ranges=full_bounds,
        seed=SEED,
    )
    tree_builder = FTC(
        scenarios=scenario_fans["scenarios"],
        num_variables_per_stage=num_variables_per_stage,
        stage_ids=stage_ids
    )
    scenario_trees = tree_builder.generate_scenario_trees(r=2, initial_stage_id_to_cluster=1)
    save_json(
        output_dir="./same_ren_scentree",
        num_stages=len(stage_ids),
        in_sample_prediction=build_in_sample_fans,
        predicted_value=scenario_fans["predicted_values"],
        observed_value=scenario_fans["observed_values"],
        scenario_trees=scenario_trees,
        mapping_datasets_columns=map_columns_names,
        multiple_files=True,
        name = f"scenariotree_{num_scenarios}"
    )

In [ ]:
#scenario_fans["scenarios"][0].shape

### Scenario Tree

In [ ]:
"""
tree_builder = FTC(
    scenarios=scenario_fans["scenarios"],
    num_variables_per_stage=num_variables_per_stage,
    stage_ids=stage_ids
)
scenario_trees = tree_builder.generate_scenario_trees(r=2, initial_stage_id_to_cluster=1)
"""

### Output

In [ ]:
"""
save_json(
    output_dir="./same_ren_scentree",
    num_stages=len(stage_ids),
    in_sample_prediction=build_in_sample_fans,
    predicted_value=scenario_fans["predicted_values"],
    observed_value=scenario_fans["observed_values"],
    scenario_trees=scenario_trees,
    mapping_datasets_columns=map_columns_names,
    multiple_files=True,
    name = f"scenariotree_{num_scenarios}"
)
"""

# Simulated data

In [ ]:
X_a = np.random.normal(size=(10, 3))
X_b = np.random.normal(size=(10, 4))
datasets = [
    Dataset(
        name="a",
        values=X_a,
        stage_ids=[1, 2, 1],
        bounds=(0, 1)
    ),
    Dataset(
        name="b",
        values=X_b,
        stage_ids=[1, 2, 3, 3],
    )
]
dataset_loader = DatasetsLoader(datasets=datasets)
full_bounds = dataset_loader.get_full_bounds()
full_values = dataset_loader.get_full_values()
num_variables_per_stage = dataset_loader.get_num_variables_per_stage()
stage_ids = dataset_loader.get_sorted_stage_ids()
map_columns_names = dataset_loader.create_stages_columns_mapping()
print(map_columns_names)

### Scenario fan

In [ ]:
num_fans = 2
num_scenarios = 3
build_in_sample_fans = True
stage_manager = StageManager()
scenario_fans = stage_manager.generate_scenario_fans(
    X=full_values,
    num_fans=num_fans,
    num_scenarios=num_scenarios,
    build_in_sample_fans=build_in_sample_fans,
    value_ranges=full_bounds
)

In [ ]:
print(scenario_fans["scenarios"])

### Scenario tree

In [ ]:
tree_builder = FTC(
    scenarios=scenario_fans["scenarios"],
    num_variables_per_stage=num_variables_per_stage,
    stage_ids=stage_ids
)
scenario_trees = tree_builder.generate_scenario_trees(r=2, initial_stage_id_to_cluster=1)

### Output

In [ ]:
save_json(
    output_dir=".",
    num_stages=len(stage_ids),
    in_sample_prediction=build_in_sample_fans,
    predicted_value=scenario_fans["predicted_values"],
    observed_value=scenario_fans["observed_values"],
    scenario_trees=scenario_trees,
    mapping_datasets_columns=map_columns_names,
    multiple_files=False,
)